# 00 · Environment smoke test (milestone M0)

Run this once before any real training. It answers four questions on the actual Colab GPU:

1. Do the libraries install and do the unit tests pass?
2. Does the 4-bit teacher load on the T4, and how fast does it generate?
3. Does the **whole pipeline** run end to end with real models on a tiny dataset?
4. How long will the full `msci-code-fast` run take, in hours and compute units?

It also writes a version lock file to Drive so later sessions install exactly the same packages.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Copy the `svlm-catalogue` folder to
`MyDrive/svlm/` (or upload it as a zip when asked). Expected cost: about 30–45 minutes, roughly 1 compute unit.

In [ ]:
# --- Setup: GPU check, Google Drive, repo, packages (run every session) ---
import glob, json, os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
SVLM_ROOT = '/content/drive/MyDrive/svlm'       # run outputs, checkpoints and the model store live here
REPO_ON_DRIVE = f'{SVLM_ROOT}/svlm-catalogue'   # copy the repo folder here once (Drive for Desktop works)
REPO = '/content/svlm-catalogue'                # working copy on Colab's fast local disk
os.environ['SVLM_ROOT'] = SVLM_ROOT
os.makedirs(SVLM_ROOT, exist_ok=True)

if not os.path.exists(REPO):
    if os.path.exists(REPO_ON_DRIVE):
        subprocess.run(['cp', '-r', REPO_ON_DRIVE, REPO], check=True)
    else:
        from google.colab import files
        print('Repo not found on Drive. Upload svlm-catalogue.zip:')
        up = files.upload()
        subprocess.run(['unzip', '-q', next(iter(up)), '-d', '/content'], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)

locks = sorted(glob.glob(f'{SVLM_ROOT}/env/lock-colab-*.txt'))
REQ = locks[-1] if locks else 'requirements/colab.txt'
print('installing from', REQ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | compute {p.major}.{p.minor} | {p.total_memory/1e9:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}')

## 1 · Unit tests (CPU logic: FIM spans, scoring, KD loss, data alignment)

In [ ]:
!python -m pytest -q tests/test_core.py

## 2 · Teacher on the T4

Loads the teacher in 4-bit (bitsandbytes NF4, fp16 compute), checks memory, and measures batched
generation throughput on real FIM prompts.

In [ ]:
import time, yaml
from svlm.modeling import load_causal_lm, load_tokenizer, generate_batch, free_gpu
from svlm.fim import fim_prompt

cfg = yaml.safe_load(open('configs/code_fast.yaml'))
name = cfg['teacher']['model']
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
tok = load_tokenizer(name)
teacher = load_causal_lm(name, load_in_4bit=True)
print(f'loaded {name} in {time.time()-t0:.0f}s, weights use {torch.cuda.memory_allocated()/1e9:.1f} GB')

prefix = "import pandas as pd\n\ndef monthly_returns(prices: pd.Series) -> pd.Series:\n    \"\"\"Month-end simple returns.\"\"\"\n"
suffix = "\n    return rets.dropna()\n"
prompts = [fim_prompt(prefix, suffix)] * 16
t0 = time.time()
outs = generate_batch(teacher, tok, prompts, max_new_tokens=128)
dt = time.time() - t0
n_new = sum(len(tok(o, add_special_tokens=False)['input_ids']) for o in outs)
print(f'batch of 16: {n_new} new tokens in {dt:.1f}s -> {n_new/dt:.0f} tokens/s | peak memory {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
print('--- sample completion ---')
print(outs[0].split('<|endoftext|>')[0])
TEACHER_TOK_PER_S = n_new / dt
free_gpu(teacher)

## 3 · Full pipeline, tiny data, real models

Runs all nine steps with 96 training and 32 held-out samples (`version=0.0.0`, `tier=smoke`), so it
never touches the real run. Gates will probably fail at this size; `register.force=true` lets the
register step run anyway so we can check the model card and catalogue entry.

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml \
    --set version=0.0.0 --set tier=smoke \
    --set data.fim.n_train=96 --set data.fim.n_heldout=32 \
    --set eval.humaneval=false --set register.force=true

## 4 · Time and compute-unit estimate for the full run

In [ ]:
from pathlib import Path
from svlm.utils import read_json

CU_PER_HOUR = 1.5   # read the real figure from Colab's Resources panel and edit this

smoke = Path(SVLM_ROOT) / 'runs' / 'msci-code-fast' / '0.0.0-smoke'
sec = {s: read_json(smoke / s / '_SUCCESS')['seconds'] for s in
       ['prepare', 'teacher_generate', 'verify', 'teacher_logits', 'train_sft', 'train_kd', 'evaluate', 'quantise', 'register']}
f = cfg['data']['fim']
n_tr_s, n_he_s = 96, 32
n_tr, n_he = f['n_train'], f['n_heldout']
aug = cfg['data']['teacher_augment_fraction']
scale = {
    'prepare': 1.5,                                                      # mostly cloning, roughly fixed
    'teacher_generate': (n_he + aug * n_tr) / (n_he_s + aug * n_tr_s),
    'verify': n_tr / n_tr_s,
    'teacher_logits': n_tr / n_tr_s,
    'train_sft': n_tr / n_tr_s,
    'train_kd': n_tr / n_tr_s,
    'evaluate': n_he / n_he_s,
    'quantise': 1.0,
    'register': 1.0,
}
est = {s: sec[s] * scale[s] / 3600 for s in sec}
for s in sec:
    print(f'{s:17s} smoke {sec[s]:7.0f}s   full est. {est[s]:5.2f} h')
total = sum(est.values()) + 0.5  # + HumanEval+ run
print(f'\nEstimated full msci-code-fast run: {total:.1f} GPU hours ≈ {total*CU_PER_HOUR:.0f} compute units')
print('Model fixed costs (loading) are included in the smoke timings, so this is a slight over-estimate.')

## 5 · Check the smoke model card

In [ ]:
from IPython.display import Markdown, display
card = Path(SVLM_ROOT) / 'model_store' / 'msci-code-fast' / '0.0.0-smoke' / 'model_card.md'
display(Markdown(card.read_text()))

## 6 · Write the version lock file (later sessions install exactly these versions)

In [ ]:
import datetime, importlib.metadata as md_
pkgs = ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'pyyaml', 'numpy', 'sentencepiece', 'evalplus', 'pytest']
lock = [f'{p}=={md_.version(p)}' for p in pkgs]
env_dir = Path(SVLM_ROOT) / 'env'; env_dir.mkdir(exist_ok=True)
path = env_dir / f'lock-colab-{datetime.date.today():%Y%m%d}.txt'
path.write_text('\n'.join(lock) + '\n')
info = {'gpu': p.name, 'capability': f'{p.major}.{p.minor}', 'torch': torch.__version__, 'cuda': torch.version.cuda,
        'teacher_tokens_per_s': round(TEACHER_TOK_PER_S), 'lock': lock}
(env_dir / 'env_report.json').write_text(json.dumps(info, indent=2))
print(path.read_text()); print(json.dumps(info, indent=2))

## Optional · vLLM teacher backend

vLLM is faster for long generations (it matters for `msci-reason` in M3) but it installs its own torch
build. Test it in a **fresh runtime** (*Runtime → Disconnect and delete runtime*), then run:

```python
!pip install -q -r requirements/vllm.txt
from vllm import LLM, SamplingParams
llm = LLM('Qwen/Qwen2.5-Coder-7B', quantization='bitsandbytes', dtype='half', max_model_len=2048, gpu_memory_utilization=0.85)
print(llm.generate(['<|fim_prefix|>def add(a, b):
<|fim_suffix|>
<|fim_middle|>'], SamplingParams(max_tokens=32))[0].outputs[0].text)
```

If that works, set `teacher.backend: vllm` for the teacher_generate step only, and restart the runtime
before training.

## Done when

- [ ] unit tests pass
- [ ] teacher loads and generates; throughput recorded
- [ ] all nine steps finish on the smoke run
- [ ] lock file written to `MyDrive/svlm/env/`